In [ ]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import tqdm
import pandas as pd
from sklearn.linear_model import LinearRegression
import seaborn as sns

import matplotlib as mpl
import matplotlib.patches as patches

from scipy.stats import ttest_1samp
from statsmodels.stats.multitest import fdrcorrection
from mne.stats import permutation_t_test
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import patches
from mne import create_info
from mne.viz import plot_topomap
from mne.channels import make_dig_montage, make_standard_montage

import pickle

from statannotations.Annotator import Annotator
from matplotlib.collections import PatchCollection
import matplotlib.patches as mpatches

In [ ]:
home_path = os.path.expanduser('~')

figure_dir = f'../figures'
os.makedirs(figure_dir, exist_ok=True)

tab10 = mpl.colormaps.get_cmap('tab10')
tab20 = mpl.colormaps.get_cmap('tab20')
tab20c = mpl.colormaps.get_cmap('tab20c')
tab20b = mpl.colormaps.get_cmap('tab20b')

feature = None

In [ ]:
sns.set(font_scale=1.5)
sns.set_style('ticks')

plt.rcParams.update({
    'axes.titlesize': 'large',
    'axes.labelsize': 'large',
    'xtick.labelsize':'large',
    'ytick.labelsize':'large'
    })

In [ ]:
import yaml
with open('../eeg_info.pkl', 'r') as f:
    eeg_info = yaml.load(f, Loader=yaml.FullLoader)

channel_names = eeg_info['main_experiment']['channel_names']
sample_rate = eeg_info['main_experiment']['sample_rate']
t = eeg_info['main_experiment']['t']
visual_channel_names = eeg_info['main_experiment']['visual_channel_names']
visual_channel_indices = eeg_info['main_experiment']['visual_channel_indices']

channel_names_res = eeg_info['additional_experiment']['channel_names']
visual_channel_indices_res = eeg_info['additional_experiment']['visual_channel_indices']

In [ ]:
# Load Noise Ceiling

with open('../eeg_data/main_experiment/noise_ceiling.pkl', 'rb') as f:
    noise_ceiling = pickle.load(f)
    
upper_bound_per_electrode = noise_ceiling['upper_bound']
lower_bound_per_electrode = noise_ceiling['lower_bound']
upper_corr_subs = noise_ceiling['upper_corr_subs']
lower_corr_subs = noise_ceiling['lower_corr_subs']

In [ ]:
visual_channel_lower_bound_average = lower_bound_per_electrode[:, visual_channel_indices].mean(axis=1)
visual_channel_upper_bound_average = upper_bound_per_electrode[:, visual_channel_indices].mean(axis=1)

In [ ]:
noise_ceiling_res = {}
mode = '_manual'

for sub in tqdm.tqdm([12, 15, 16, 17, 18, 19], total=6):
    for cond in ['center', 'peri', 'size']:
        for cond_index in range(1, 4):
            
            if f'{cond}{cond_index}' not in noise_ceiling_res:
                noise_ceiling_res[f'{cond}{cond_index}'] = {sub: {}}
            else:
                noise_ceiling_res[f'{cond}{cond_index}'][sub] = {}

            eeg_dir = f"../eeg_data/additional_experiment"
            _lower = np.load(os.path.join(eeg_dir, f'sub{str(sub).zfill(3)}', f'sub{str(sub).zfill(3)}_{cond}{cond_index}_lower_noise_ceiling.npy'))
            _upper = np.load(os.path.join(eeg_dir, f'sub{str(sub).zfill(3)}', f'sub{str(sub).zfill(3)}_{cond}{cond_index}_upper_noise_ceiling.npy'))

            noise_ceiling_res[f'{cond}{cond_index}'][sub]['lower'] = _lower
            noise_ceiling_res[f'{cond}{cond_index}'][sub]['upper'] = _upper

In [ ]:
for sub in range(1, 5):
    for cond in ['center', 'peri', 'size']:
        for cond_index in range(1, 4):
            eeg_dir = '../eeg_data/additional_experiment'
            _lower = np.load(os.path.join(eeg_dir, f'sub{str(sub).zfill(3)}', f'sub{str(sub).zfill(3)}_{cond}{cond_index}_lowerbound.npy'))
            _upper = np.load(os.path.join(eeg_dir, f'sub{str(sub).zfill(3)}', f'sub{str(sub).zfill(3)}_{cond}{cond_index}_upperbound.npy'))

            noise_ceiling_res[f'{cond}{cond_index}'][sub] = {'lower': _lower, 'upper': _upper}
            # noise_ceiling_res[f'{cond}{cond_index}'][sub] = {'upper': _upper}

In [ ]:
visual_channel_lower_bound_average_res = {
    key: np.array([noise_ceiling_res[key][sub]['lower'][visual_channel_indices_res].mean(axis=0) for sub in noise_ceiling_res[key]])
    for key in noise_ceiling_res.keys() 
}
visual_channel_upper_bound_average_res = {
    key: np.array([noise_ceiling_res[key][sub]['upper'][visual_channel_indices_res].mean(axis=0) for sub in noise_ceiling_res[key]])
    for key in noise_ceiling_res.keys()
}

In [ ]:
def get_circular_mask(size, center_fraction):
    # center_fraction = np.sqrt(center_fraction)

    # Calculate the area of the mask
    mask_area = size[0] * size[1]

    # Calculate the center size based on the percentage of the area
    center_area = mask_area * center_fraction

    # Calculate the radius of the circular center
    center_radius = int(np.sqrt(center_area / np.pi))

    # # Calculate the padding needed to center the circular center within the mask
    # padding = (size - center_radius) // 2

    # Create the mask
    mask = np.zeros((size[0], size[1]), dtype=int)
    y, x = np.ogrid[:size[0], :size[1]]
    mask[((x - size[1] // 2) ** 2 + (y - size[0] // 2) ** 2) < center_radius ** 2] = 1

    return mask

def get_rect_mask(shape, fraction):
    # center_fraction = fraction
    fraction = np.sqrt(fraction)

    mask_small = np.zeros(shape).astype(int)

    row_center = shape[0] / 2
    row_start = int(np.floor(row_center - row_center * fraction))
    row_end = int(np.ceil(row_center + row_center * fraction))
    # row_size = row_end - row_start
    # print(row_start, row_end)

    col_center = shape[1] / 2
    col_start = int(np.floor(col_center - col_center * fraction))
    col_end = int(np.ceil(col_center + col_center * fraction))
    # col_size = col_end - col_start
    # print(col_start, col_end)
    
    mask_small[row_start:row_end, col_start:col_end] = 1

    return mask_small

def make_custom_info(topoplot_channel_names, x_spacing = 0.015, y_spacing = 0.085, z_spacing = 0.0):

    info = create_info(ch_names=make_standard_montage('biosemi64').ch_names, sfreq=1024, ch_types='eeg')
    info.set_montage('biosemi64')

    position_dict = info.get_montage().get_positions()
    position_dict['ch_pos'].update({'I1': np.array([x_spacing, -y_spacing, z_spacing]), 'I2': np.array([-x_spacing, -y_spacing, z_spacing])})
    position_dict['ch_pos'].pop('F5')
    position_dict['ch_pos'].pop('F6')
    custom_montage = make_dig_montage(**position_dict)


    topoplot_channel_names = ['I1' if x == 'F5' else ('I2' if x == 'F6' else x) for x in topoplot_channel_names]# + ['I1', 'I2']
    # if 'I1' not in topoplot_channel_names:
    #     topoplot_channel_names.append('I1')
    # if 'I2' not in topoplot_channel_names:
    #     topoplot_channel_names.append('I2')

    info = create_info(ch_names=topoplot_channel_names, sfreq=1024, ch_types='eeg')
    info.set_montage(custom_montage)

    # fig = custom_montage.plot()

    return info, topoplot_channel_names


def fdr_correct_per_timepoint(tvalues, timepoints):
    _, corrected_pvals = fdrcorrection([tvalues[timepoint_index].pvalue for timepoint_index in timepoints])

    l = {timepoint_index: corrected_pvals[counter] for counter, timepoint_index in enumerate(timepoints)}

    return l

def fdr_correct_per_channel_timepoint(tvalues, timepoints, channels):
    _, corrected_pvals = fdrcorrection([tvalues[(timepoint_index, channel)].pvalue for timepoint_index in timepoints for channel in channels])
    l = {}
    counter = 0
    for timepoint_index in timepoints:
        _l = {}
        for channel in channels:
            _l[channel] = corrected_pvals[counter]
            counter += 1
        
        l[timepoint_index] = _l

    return l

def plot_tvalue_topoplots(axes, tvalues, corrected_pvals, timepoints, channels, vlim, t, alpha=0.05, cbar_ax=None, legend_label='t-value fdr-corrected (significant with alpha = {alpha})', names=True, title:bool=True, title_fontsize=24, axis_title_pad=6.0, mask=None, mask_params=None):
    global_min_sig_t_im = None
    global_min_sig_t = np.inf

    for ax_index, timepoint_index in enumerate(timepoints):
        tvalue_data = []
        tvalue_channel_names = []

        min_sig_t = np.inf
        for channel in channels:
            if channel in ['left', 'right', 'above', 'below']:
                continue
            _stat = tvalues[(timepoint_index, channel)]
            tvalue_data.append(_stat.statistic)
            # print(_stat.pvalue)
            if corrected_pvals[timepoint_index][channel] < alpha and np.abs(_stat.statistic) < np.abs(min_sig_t):
                min_sig_t = _stat.statistic
            tvalue_channel_names.append(channel)
        # print(min_sig_t)
        info, tvalue_channel_names = make_custom_info(tvalue_channel_names)

        p = [vlim[0], -np.abs(min_sig_t), np.abs(min_sig_t), vlim[1]]
        f = lambda x: np.interp(x, p, [0, 0.5, 0.5, 1])

        # cmap = 'seismic'
        cmap = LinearSegmentedColormap.from_list('map_white', 
                    list(zip(np.linspace(0,1), plt.cm.seismic(f(np.linspace(min(p), max(p)))))))
        im,_ = plot_topomap(tvalue_data, info, axes=axes[ax_index], names=tvalue_channel_names if type(names) is bool and names else (names if type(names) is list and ax_index == 0 else None), 
                            vlim=vlim, show=False, cmap=cmap, mask=mask, mask_params=mask_params) #, image_interp='nearest')
        if np.abs(min_sig_t) < np.abs(global_min_sig_t):
            global_min_sig_t_im = im

        if title:
            axes[ax_index].set_title(f'{int(t[timepoint_index]*1000):d} ms', pad=axis_title_pad, fontsize=title_fontsize)

    # ax_x_start = 0.95
    # ax_x_width = 0.04
    # ax_y_start = 0.1
    # ax_y_height = 0.8

    # # cbar_ax = fig.add_axes([ax_x_start, ax_y_start, ax_x_width, ax_y_height])
    if cbar_ax is not None and global_min_sig_t_im is not None:
        cbar = plt.colorbar(global_min_sig_t_im, cax=cbar_ax)
        cbar.set_label(legend_label.format(alpha=alpha))

    # fig.suptitle('T-Test: RGB > COC?')

    # plt.show()
    # return axes

In [ ]:
with open('../example_images/example_features.pkl', 'rb') as f:
    d = pickle.load(f)
    feature = d['feature']
    gcs_feature = d['gcs_feature']

def get_feature_axes(ax, set_title, axis_off:bool=True, cmap='gray', fraction=0.05, is_circle:bool=False):
    
    shape = feature.shape
    # fraction = np.sqrt(0.005)
    _fraction = np.sqrt(fraction)
    row_center = shape[0] / 2
    row_start = int(row_center - row_center * _fraction)
    row_end = int(row_center + row_center * _fraction)
    row_size = row_end - row_start

    col_center = shape[1] / 2
    col_start = int(col_center - col_center * _fraction)
    col_end = int(col_center + col_center * _fraction)
    col_size = col_end - col_start

    # mask = np.zeros(shape).astype(bool)
    # mask[row_start:row_end, col_start:col_end] = True
    # whole_size = shape[0]*shape[1]
    mask = get_rect_mask(shape, fraction).astype(bool) if not is_circle else get_circular_mask(shape, fraction).astype(bool)


    # Full Scene
    ax[0].imshow(feature, cmap=cmap)
    brown_patch = mpatches.Rectangle((-1, -1), shape[1]+1, shape[0]+1, edgecolor=tab10.colors[5], facecolor='none', linewidth=8, clip_on=False, zorder=100)
    ax[0].add_patch(brown_patch)

    # Center
    ax[1].imshow(np.where(mask, feature, feature.max()), cmap=cmap)
    if is_circle:
        orange_patch = mpatches.Circle((col_center, row_center), radius=row_size/2+2, edgecolor=tab10.colors[1], facecolor='none', linewidth=8, zorder=100)
    else:
        orange_patch = mpatches.Rectangle((col_start-2, row_start-2), col_size+3, row_size+3, edgecolor=tab10.colors[1], facecolor='none', linewidth=8, zorder=100)
    ax[1].add_patch(orange_patch)

    # Periphery
    mask = ~mask
    ax[2].imshow(np.where(mask, feature, feature.max()), cmap=cmap)
    
    if is_circle:
        blue_patch = mpatches.Circle((col_center, row_center), radius=row_size/2+2, edgecolor=tab10.colors[0], facecolor='none', linewidth=8, zorder=100)
    else:
        blue_patch = mpatches.Rectangle((col_start-2, row_start-2), col_size+3, row_size+3, edgecolor=tab10.colors[0], facecolor='none', linewidth=8, zorder=100)
    
    blue_surround_patch = mpatches.Rectangle((-1, -1), shape[1]+1, shape[0]+1, edgecolor=tab10.colors[0], facecolor='none', linewidth=8, clip_on=False, zorder=100)
    ax[2].add_patch(blue_patch)
    ax[2].add_patch(blue_surround_patch)

    # # GCS
    ax[3].imshow(gcs_feature(feature), cmap=cmap)
    brown_patch = mpatches.Rectangle((-1, -1), shape[1]+1, shape[1]+1, edgecolor='black', facecolor='none', linewidth=8, clip_on=False, zorder=100)
    ax[3].add_patch(brown_patch)

    if axis_off:
        for _ax in ax.flatten():
            _ax.axis('off')
    else:
        for _ax in ax.flatten():
            _ax.set_xticks([])
            _ax.set_yticks([])

    if set_title:
        ax[0].set_title('Full', fontsize=28)
        ax[1].set_title(f'Center {fraction:.1%}', fontsize=28)
        ax[2].set_title(f'Periphery {1-fraction:.1%}', fontsize=28)
        ax[3].set_title('GCS', fontsize=28)


In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(36, 12), gridspec_kw={'wspace': 0.05})

get_feature_axes(ax, set_title=False, axis_off=False, cmap='gray', fraction=0.005, is_circle=True)

fig.savefig(os.path.join(figure_dir, 'Fig-1_spatial_sampling_inset.png'), dpi=300, bbox_inches='tight')

plt.show()

# Figure 2

In [ ]:
_main_data = pd.read_parquet('../results/alexnet_imagenet_encoding_results.parquet')
_main_data.subject = _main_data.subject.astype(int)
_main_data.crop_condition = _main_data.crop_condition.astype('category')
_main_data.crop_instance = _main_data.crop_instance.astype('category')
_main_data.channel = _main_data.channel.astype('category')
_main_data.fraction = _main_data.fraction.astype('category')
_main_data.drop(columns=['pred_index'], inplace=True)
_main_data = _main_data[(_main_data['metric'] == 'test_corr_train')].copy()

In [ ]:
# fig-1 encoding model performance inset

fig, ax = plt.subplots(1, 1, figsize=(15, 6))

_data = _main_data[((_main_data['crop_instance'] == 'gcs-full') & (_main_data['channel'] == 'Iz'))].copy().reset_index()
f = sns.lineplot(data=_data, x='timepoint', y='value', ax=ax, linewidth=3, color=tab20c.colors[12])

ax.axvline(0, color='gray', linestyle='--')
ax.axhline(0, color='gray', linestyle='--')

channels = [x for x in channel_names if x not in ['I1', 'I2', 'F5', 'F6', 'above', 'below', 'left', 'right']]
channel_mask = np.array([x in visual_channel_names for x in channels])
mask_params = dict(marker='o', markerfacecolor='k', markeredgecolor='k',
        linewidth=0, markersize=6)

info, tvalue_channel_names = make_custom_info(channels)
mask_params = dict(marker='o', markerfacecolor='k', markeredgecolor='k',
        linewidth=0, markersize=6)

inset = ax.inset_axes([0.65, 0.55, 0.3, 0.3])
_ = plot_topomap(data=np.zeros((len(channels), )), pos=info, axes=inset, mask_params=mask_params, mask=channel_mask, sensors=False, show=False)

ax.set_ylabel('Encoding performance (r)', fontsize=32, fontweight='bold')
ax.set_xlabel('Time (s)', fontsize=32, fontweight='bold')
# limit number of x-ticklabels
ax.xaxis.set_major_locator(plt.MaxNLocator(2))
# ax.set_xticks([0.0, 0.1, 0.2, 0.3])
ax.yaxis.set_major_locator(plt.MaxNLocator(3))

ax.spines[['right', 'top']].set_visible(False)


ax.fill_between(t, lower_bound_per_electrode[:, channel_names.index('Iz')], upper_bound_per_electrode[:, channel_names.index('Iz')], color='gray', alpha=0.3, label='Noise Ceiling')
ax.legend(frameon=False, fontsize=28, bbox_to_anchor=(0.01, 1.15), loc='upper left')

# fig.savefig(os.path.join(figure_dir, 'Fig-1_encoding_model_inset.png'), dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
def get_fdr_correct_pvals(_data, channels, crop_instances=['feature-full', 'center', 'periphery', 'gcs-full']):
    # channel = 'Iz'
    _tval_data = _data[(_data['crop_instance'].isin(crop_instances)) & (_data['channel'].isin(channels))].copy()
    _tval_data.channel = _tval_data.channel.cat.remove_unused_categories()
    _tval_data = _tval_data.groupby(['timepoint', 'subject', 'crop_instance', 'channel']).value.mean().reset_index()
    
    # timepoints = _tval_data[_tval_data.timepoint > 0.0].timepoint.unique()
    # crop_instances = _tval_data.crop_instance.unique()

    _tval_data.crop_instance = _tval_data.crop_instance.cat.reorder_categories(crop_instances, ordered=True)
    _tval_data.channel = _tval_data.channel.cat.reorder_categories(_tval_data.channel.unique(), ordered=True)

    _tval_data.sort_values(['subject', 'crop_instance', 'timepoint', 'channel'], inplace=True)
    vals = _tval_data.value.values
    n_crop_instances = len(crop_instances)
    n_timepoints = _tval_data.timepoint.nunique()
    n_channels = _tval_data.channel.nunique()

    vals = vals.reshape(-1, n_crop_instances*n_timepoints*n_channels)

    stats = permutation_t_test(vals, n_permutations=10000, n_jobs=10)

    pvals = stats[1].reshape(n_crop_instances, n_timepoints, n_channels)

    condition_stats = {}
    for index, channel in enumerate(_tval_data.channel.unique()):
        condition_stats[channel] = {}
        for crop_instance_index, crop_instance in enumerate(crop_instances):
            condition_stats[channel][crop_instance] = {}
            for timepoint_index, timepoint in enumerate(_tval_data.timepoint.unique()):
                condition_stats[channel][crop_instance][timepoint] = pvals[crop_instance_index, timepoint_index, index]

    return condition_stats

In [ ]:
#### Prepare plot #####

channels = [x for x in channel_names if x not in ['I1', 'I2', 'F5', 'F6', 'above', 'below', 'left', 'right']]
channel_mask = np.array([x in ['Iz', 'Pz'] for x in channels])
mask_params = dict(marker='o', markerfacecolor='k', markeredgecolor='k',
        linewidth=0, markersize=6)
plot_channel_names = None # [x if channel_mask[i] else '' for i, x in enumerate(channels)]


timepoints = [200, 225, 250]
do_channels = ['Iz', 'Pz']

hue_order = ['feature-full', 'center', 'periphery', 'gcs-full']

colormaps = {
    'feature-full': tab10.colors[5],
    'gcs-full': 'black', #tab10.colors[0],
    'center': tab10.colors[1],
    'periphery': tab10.colors[0], #tab10.colors[7],
}

linewidth = 3
n_rows = 6
n_cols = 8 # 3 # len(timepoints) + 4

info, tvalue_channel_names = make_custom_info(channels)
mask_params = dict(marker='o', markerfacecolor='k', markeredgecolor='k',
        linewidth=0, markersize=6)

In [ ]:
###################### STATS
pval_colors = [colormaps[x] for x in hue_order]
condition_stats = get_fdr_correct_pvals(_main_data, channels=do_channels, crop_instances=hue_order)


In [ ]:

#### Full vs Center
all_tvalues = {}
tvalues = {}
for timepoint_index in timepoints:
    for channel in tqdm.tqdm(channels):
        ttest_data = _main_data[(_main_data['timepoint_index'] == timepoint_index) & (_main_data['channel'] == channel)]

        ttest_data_full = ttest_data[(ttest_data['crop_instance'] == 'feature-full')].groupby('subject').value.mean().to_list() #[:22]
        ttest_data_center = ttest_data[(ttest_data['crop_instance'] == 'center') & (ttest_data['fraction'] == main_fraction)].groupby('subject').value.mean().to_list() #[:22]

        diff = np.array(ttest_data_full) - np.array(ttest_data_center)
        res = ttest_1samp(diff, popmean=0)

        tvalues[(timepoint_index, channel)] = res
all_tvalues['full_vs_center'] = tvalues

#### Full vs. Periphery
tvalues = {}
for timepoint_index in timepoints:
    for channel in tqdm.tqdm(channels):
        ttest_data = _main_data[(_main_data['timepoint_index'] == timepoint_index) & (_main_data['channel'] == channel)]

        ttest_data_full = ttest_data[(ttest_data['crop_instance'] == 'feature-full')].groupby('subject').value.mean().to_list() #[:22]
        ttest_data_periphery = ttest_data[(ttest_data['crop_instance'] == 'periphery') & (ttest_data['fraction'] == main_fraction)].groupby('subject').value.mean().to_list() #[:22]

        diff = np.array(ttest_data_full) - np.array(ttest_data_periphery)
        res = ttest_1samp(diff, popmean=0)

        tvalues[(timepoint_index, channel)] = res

all_tvalues['full_vs_periphery'] = tvalues

#### GCS vs Full
tvalues = {}
for timepoint_index in timepoints:
    for channel in tqdm.tqdm(channels):
        ttest_data = _main_data[(_main_data['timepoint_index'] == timepoint_index) & (_main_data['channel'] == channel)]

        ttest_data_gcs = ttest_data[(ttest_data['crop_instance'] == 'gcs-full')].groupby('subject').value.mean().to_list() #[:22]
        ttest_data_full = ttest_data[(ttest_data['crop_instance'] == 'feature-full')].groupby('subject').value.mean().to_list() #[:22]

        diff = np.array(ttest_data_gcs) - np.array(ttest_data_full)
        res = ttest_1samp(diff, popmean=0)

        tvalues[(timepoint_index, channel)] = res

all_tvalues['gcs_vs_full'] = tvalues

#### GCS vs. Center-Crop
tvalues = {}
for timepoint_index in timepoints:
    for channel in tqdm.tqdm(channels):
        ttest_data = _main_data[(_main_data['timepoint_index'] == timepoint_index) & (_main_data['channel'] == channel)]

        ttest_data_gcs = ttest_data[(ttest_data['crop_instance'] == 'gcs-full')].groupby('subject').value.mean().to_list()
        ttest_data_center = ttest_data[(ttest_data['crop_instance'] == 'center') & (ttest_data['fraction'] == main_fraction)].groupby('subject').value.mean().to_list()

        diff = np.array(ttest_data_gcs) - np.array(ttest_data_center)
        res = ttest_1samp(diff, popmean=0)

        tvalues[(timepoint_index, channel)] = res

all_tvalues['gcs_vs_center'] = tvalues
_x = []
for key, values in all_tvalues.items():
    _x.extend([x.statistic for x in values.values()])
vlim = max(np.abs(min(_x)), np.abs(max(_x)))
vlim = (-vlim, vlim)

channel_mask = np.array([False for x in channels])

corrected_pvals = {cond: fdr_correct_per_channel_timepoint(tvalues=all_tvalues[cond], timepoints=timepoints, channels=channels) for cond in all_tvalues.keys()}


In [ ]:
print('Iz\n')
print('full', min([timepoint for timepoint, x in condition_stats['Iz']['feature-full'].items() if x < 0.05]))
print('center', min([timepoint for timepoint, x in condition_stats['Iz']['center'].items() if x < 0.05]))
print('periphery', min([timepoint for timepoint, x in condition_stats['Iz']['periphery'].items() if x < 0.05]))
print('gcs-full', min([timepoint for timepoint, x in condition_stats['Iz']['gcs-full'].items() if x < 0.05]))



print('\n\nPz\n')
print('full', min([timepoint for timepoint, x in condition_stats['Pz']['feature-full'].items() if x < 0.05]))
print('center', min([timepoint for timepoint, x in condition_stats['Pz']['center'].items() if x < 0.05]))
print('periphery', min([timepoint for timepoint, x in condition_stats['Pz']['periphery'].items() if x < 0.05]))
print('gcs-full', min([timepoint for timepoint, x in condition_stats['Pz']['gcs-full'].items() if x < 0.05]))

In [ ]:
fig = plt.figure(layout='constrained', figsize=(35, 18)) # 24

subfigs = fig.subfigures(1, 2, width_ratios=[2, 7], wspace=0.05) # 
left_subfigs = subfigs[0].subfigures(2, 1)
feature_plots = left_subfigs[0].subplots(2, 2, gridspec_kw={'height_ratios': [2, 3], 'width_ratios': [2, 2]}).flatten()

bar_ax = left_subfigs[1].subplots(1, 1)

bar_data = _main_data.groupby(['subject', 'crop_instance']).value.mean().reset_index().sort_values(['subject', 'crop_instance'])
bar_data['crop_instance'] = bar_data['crop_instance'].replace({'feature-full': 'Full', 'center': 'Center', 'periphery': 'Periphery', 'gcs-full': 'GCS'})

colormaps = {
    'Full': tab10.colors[5],
    'GCS': 'black', 
    'Center': tab10.colors[1],
    'Periphery': tab10.colors[0], 
}

plotting_parameters = {
    'data':    bar_data,
    'x':       'crop_instance',
    'y':       'value',
    'palette': colormaps,
    'order':    ['Full', 'Center', 'Periphery', 'GCS'],
}

f = sns.barplot(ax=bar_ax, **plotting_parameters)

pairs = [
    ('Full', 'Center'),
    ('Full', 'Periphery'),
    ('Full', 'GCS'),
    ('Center', 'Periphery'),
    ('Center', 'GCS'),
    ('Periphery', 'GCS')
]

annotator = Annotator(pairs=pairs, ax=bar_ax, **plotting_parameters, verbose=True)
annotator.configure(test='Wilcoxon', hide_non_significant=True, pvalue_thresholds=[[1e-3, "***"], [1e-2, "**"], [0.05, "*"]])
annotator.apply_and_annotate()

bar_ax.set_xlabel('Encoding model', fontsize=30)
bar_ax.set_ylabel('Encoding performance (r)', fontsize=30)
# limit number of yticks to 4
bar_ax.yaxis.set_major_locator(plt.MaxNLocator(4))
#######################

subfigsnest = subfigs[1].subfigures(2, 1, height_ratios=[2, 2], hspace=0.2)
lineplots = subfigsnest[0].subplots(1, 2, sharey=True).flatten()

topoplot_subfigs = subfigsnest[1].subfigures(1, 2, wspace=0.1)

axsnest2 = topoplot_subfigs[0].subplots(2, 4, sharey=True, gridspec_kw={'wspace': -0.2, 'width_ratios': [9, 9, 9, 1]})
cbar_ax1 = axsnest2[:, -1]
gs = cbar_ax1[0].get_gridspec()
for _ax in cbar_ax1:
    _ax.remove()
cbar_ax1 = fig.add_subplot(gs[:, -1])
axsnest2 = axsnest2[:, :-1]

axsnest3 = topoplot_subfigs[1].subplots(2, 4, sharey=True, gridspec_kw={'wspace': -0.2, 'width_ratios': [9, 9, 9, 1]})
cbar_ax2 = axsnest3[:, -1]
gs = cbar_ax2[0].get_gridspec()
for _ax in cbar_ax2:
    _ax.remove()
cbar_ax2 = fig.add_subplot(gs[:, -1])
axsnest3 = axsnest3[:, :-1]

colormaps = {
    'feature-full': tab10.colors[5],
    'gcs-full': 'black', #tab10.colors[0],
    'center': tab10.colors[1],
    'periphery': tab10.colors[0], #tab10.colors[7],
}

counter = 0
for ch_index, channel in enumerate(do_channels):
    _ax = lineplots[counter]
    counter += 1

    # ================ channel insets
    iax = _ax.inset_axes([0.1, 0.6, .25, .25])
    iax.set_aspect('equal', anchor="NW")
    channel_mask = np.array([x in [channel] for x in channels])

    iax.axis('off')
    _ = plot_topomap(data=np.zeros((len(channels), )), pos=info, axes=iax, mask_params=mask_params, mask=channel_mask, sensors=False, show=False)
    iax.set_title(channel, fontweight='bold', fontsize=28)
    # ==============================


    if ch_index == 0:
        _ax.set_ylabel('Encoding performance (r)', fontsize=30)
        _ax.set_xlabel('Time (s)', fontsize=30)
    else:
        _ax.set_ylabel(' ')
        _ax.set_xlabel('Time (s)', fontsize=30)

    if channel == 'Average':
        _this_here_data = _main_data[(_main_data['crop_instance'].isin(hue_order))].reset_index()
    else:
        _this_here_data = _main_data[(_main_data['channel'] == channel) & (_main_data['crop_instance'].isin(hue_order))].reset_index()

    f = sns.lineplot(data=_this_here_data, 
                    ax=_ax, x='timepoint', y='value', hue='crop_instance', #errorbar='se', 
                    hue_order=hue_order,
                    linewidth=linewidth,
                    palette=colormaps,
                    )
    
    f.legend().set_visible(False)
    _ax.vlines(0.0, linestyles='dashed', ymin=0.0, ymax=0.8, colors='gray', alpha=0.5)
    
    if channel == 'Average':
        _ax.fill_between(t, lower_bound_per_electrode.mean(axis=1), upper_bound_per_electrode.mean(axis=1), color='gray', alpha=0.3, edgecolor='white')

    else:
        oads_ch_index = channel_names.index(channel)
        _ax.fill_between(t, lower_bound_per_electrode[:, oads_ch_index], upper_bound_per_electrode[:, oads_ch_index], color='gray', alpha=0.3, edgecolor='white')

    for timepoint in _main_data[_main_data.timepoint > -0.1].timepoint.unique():
        for cond_index, condition in enumerate(hue_order):
            if condition_stats[channel][condition][timepoint] < 0.05:
                _ax.plot(timepoint, -0.08-cond_index*0.02, '*', color=colormaps[condition])
    ######################

# #########################################
get_feature_axes(np.array(feature_plots), set_title=True, axis_off=False, fraction=main_fraction, is_circle=True)
#########################################

channel_mask = np.array([False for x in channels])

topoplot_axes = axsnest2[0, :]
topoplot_axes[0].annotate('Center', xy=(-0.1, 0.13), rotation=90, fontsize=28, fontweight='bold', backgroundcolor=('blue', 0.5), annotation_clip=False, xycoords='axes fraction', ha='center', va='bottom')
topoplot_axes[0].annotate('<', xy=(-0.1, 0.55), rotation=90, fontsize=28, fontweight='bold', annotation_clip=False, xycoords='axes fraction', ha='center', va='bottom')
topoplot_axes[0].annotate('Full', xy=(-0.1, 0.65), rotation=90, fontsize=28, fontweight='bold', backgroundcolor=('red', 0.5), annotation_clip=False, xycoords='axes fraction', ha='center', va='bottom')
plot_tvalue_topoplots(axes=topoplot_axes, tvalues=all_tvalues['full_vs_center'], corrected_pvals=corrected_pvals['full_vs_center'], timepoints=timepoints, 
                      channels=channels, vlim=vlim, t=t, alpha=0.05, legend_label='', names=plot_channel_names, axis_title_pad=30, title_fontsize=28, mask=channel_mask, mask_params=mask_params)

topoplot_axes = axsnest2[1, :]
topoplot_axes[0].annotate('Periphery', xy=(-0.1, 0.05), rotation=90, fontsize=28, fontweight='bold', backgroundcolor=('blue', 0.5), annotation_clip=False, xycoords='axes fraction', ha='center', va='bottom')
topoplot_axes[0].annotate('<', xy=(-0.1, 0.65), rotation=90, fontsize=28, fontweight='bold', annotation_clip=False, xycoords='axes fraction', ha='center', va='bottom')
topoplot_axes[0].annotate('Full', xy=(-0.1, 0.75), rotation=90, fontsize=28, fontweight='bold', backgroundcolor=('red', 0.5), annotation_clip=False, xycoords='axes fraction', ha='center', va='bottom')
plot_tvalue_topoplots(axes=topoplot_axes, tvalues=all_tvalues['full_vs_periphery'], corrected_pvals=corrected_pvals['full_vs_periphery'], timepoints=timepoints, 
                      legend_label='t-value fdr-corrected\n(significant with alpha = {alpha})',
                      channels=channels, vlim=vlim, t=t, alpha=0.05, cbar_ax=cbar_ax1, names=plot_channel_names, title=False, mask=channel_mask, mask_params=mask_params)

topoplot_axes = axsnest3[0, :]
topoplot_axes[0].annotate('Full', xy=(-0.1, 0.2), rotation=90, fontsize=28, fontweight='bold', backgroundcolor=('blue', 0.5), annotation_clip=False, xycoords='axes fraction', ha='center', va='bottom')
topoplot_axes[0].annotate('<', xy=(-0.1, 0.45), rotation=90, fontsize=28, fontweight='bold', annotation_clip=False, xycoords='axes fraction', ha='center', va='bottom')
topoplot_axes[0].annotate('GCS', xy=(-0.1, 0.55), rotation=90, fontsize=28, fontweight='bold', backgroundcolor=('red', 0.5), annotation_clip=False, xycoords='axes fraction', ha='center', va='bottom')
plot_tvalue_topoplots(axes=topoplot_axes, tvalues=all_tvalues['gcs_vs_full'], corrected_pvals=corrected_pvals['gcs_vs_full'], timepoints=timepoints, 
                      channels=channels, vlim=vlim, t=t, alpha=0.05, names=plot_channel_names, axis_title_pad=30, title_fontsize=28, mask=channel_mask, mask_params=mask_params)


topoplot_axes = axsnest3[1, :]
topoplot_axes[0].annotate('Center', xy=(-0.1, 0.05), rotation=90, fontsize=28, fontweight='bold', backgroundcolor=('blue', 0.5), annotation_clip=False, xycoords='axes fraction', ha='center', va='bottom')
topoplot_axes[0].annotate('<', xy=(-0.1, 0.48), rotation=90, fontsize=28, fontweight='bold', annotation_clip=False, xycoords='axes fraction', ha='center', va='bottom')
topoplot_axes[0].annotate('GCS', xy=(-0.1, 0.58), rotation=90, fontsize=28, fontweight='bold', backgroundcolor=('red', 0.5), annotation_clip=False, xycoords='axes fraction', ha='center', va='bottom')
plot_tvalue_topoplots(axes=topoplot_axes, tvalues=all_tvalues['gcs_vs_center'], corrected_pvals=corrected_pvals['gcs_vs_center'], timepoints=timepoints, 
                        legend_label='t-value fdr-corrected\n(significant with alpha = {alpha})',
                        channels=channels, vlim=vlim, t=t, alpha=0.05, cbar_ax=cbar_ax2, names=plot_channel_names, title=False, mask=channel_mask, mask_params=mask_params)

handles, lables = [], []
for _ax in lineplots:
    _handles, _labels = _ax.get_legend_handles_labels()
    
    for handle, label in zip(_handles, _labels):
        if label == 'feature-full':
            label = 'Full'
        elif 'center' in label:
            label = f'Center {main_fraction:.1%}'
        elif 'periphery' in label:
            label = f'Periphery {1-main_fraction:.1%}'
        elif 'gcs' in label:
            label = 'GCS'

        if label not in lables: 
            handles.append(handle)
            lables.append(label)

lables.append('Noise Ceiling')
gray_patch = mpatches.Patch(edgecolor='lightgray', facecolor='lightgray')
handles.append(gray_patch)

legend = fig.legend(handles,lables, title='', loc='upper center', bbox_to_anchor=(0.6, 1.05), ncol=len(handles), frameon=False, prop={'size': 30}) # all_axes[0]
for legobj in legend.legend_handles:
    legobj.set_linewidth(5.0)

lineplots[0].set_zorder(100)

sns.despine()

for _ax in feature_plots:
    _ax.spines[['top', 'right']].set_visible(True)

fig.text(-0.02, 1.02, 'A', color='black', size=48, weight='bold', clip_on=False) # , in_layout=False)
fig.text(0.05, 1.01, 'Encoding models', color='black', size=30, weight='bold', clip_on=False) # , in_layout=False)
fig.text(-0.02, 0.46, 'B', color='black', size=48, weight='bold', clip_on=False) # , in_layout=False)
fig.text(0.22, 1.02, 'C', color='black', size=48, weight='bold', clip_on=False) # , in_layout=False)
fig.text(0.22, 0.46, 'D', color='black', size=48, weight='bold', clip_on=False) # , in_layout=False)
fig.text(0.6, 0.46, 'E', color='black', size=48, weight='bold', clip_on=False) # , in_layout=False)

fig.savefig(os.path.join(figure_dir, f'Fig-2_imagenet_lineplots_fig2.pdf'), dpi=300., bbox_inches='tight')

plt.show()

# Figure 3

In [ ]:
def rename(x):
    if 'feature' in x:
        return 'feature'
    elif 'gcs' in x:
        return x
    else:
        return x.replace('fraction_', '').replace('_circ', '').split('_')[0]


In [ ]:
partial_all_fig_data = pd.read_parquet('../results/alexnet_imagenet_partial_correlation_results.parquet')

In [ ]:
partial_all_fig_data['condition_fraction'] = partial_all_fig_data['condition'].apply(lambda x: f"{float(x.split('_')[-1]):.1%}" if 'fraction' in x else None)
partial_all_fig_data['given_fraction'] = partial_all_fig_data['given'].apply(lambda x: f"{float(x.split('_')[-1]):.1%}" if 'fraction' in x else None)

In [ ]:
partial_all_fig_data['fraction'] = partial_all_fig_data.apply(lambda x: x.given_fraction if x.given_fraction is not None else x.condition_fraction, axis=1)

In [ ]:
partial_all_fig_data['description'] = partial_all_fig_data['condition'].astype(str).apply(lambda x: rename(x)) + ' | ' + partial_all_fig_data['given'].astype(str).apply(lambda x: rename(x)) # .apply(lambda x: rename(str(x.condition)) + ' | ' + rename(str(x.given)), axis=1)

partial_all_fig_data['row'] = partial_all_fig_data.description.apply(lambda x: 'center | periphery' if 'center' in x and 'periphery' in x
                                                        else ('gcs | feature' if 'gcs' in x and 'feature' in x
                                                        else ('center | gcs' if 'center' in x and 'gcs' in x
                                                        else ('periphery | gcs' if 'periphery' in x and 'gcs' in x else ""))))


In [ ]:
fig3_data = partial_all_fig_data[partial_all_fig_data['fraction'].isin(['0.5%', None])].copy()
fig3_data = fig3_data[fig3_data['channel'].isin(['Iz', 'Pz'])] # 'TP7', 'TP8'
fig3_data.channel = fig3_data.channel.cat.remove_unused_categories()
fig3_data.reset_index(level=0, inplace=True)

In [ ]:
hue_pairs = [['center | periphery', 'periphery | center'], ['gcs | feature', 'feature | gcs'], ['center | gcs', 'gcs | center'], ['periphery | gcs', 'gcs | periphery']]

fig3_all_pvals = {}
for hues in hue_pairs:
    _data = fig3_data[(fig3_data['channel'].isin(['Iz', 'Pz']))].copy()
    _data['channel'] = _data['channel'].astype('category').cat.reorder_categories(['Iz', 'Pz'])

    _data = _data[(_data['description'].isin(hues))].copy()
    _data['description'] = _data['description'].astype('category').cat.reorder_categories(hues)

    _data = _data.sort_values(['description', 'channel', 'timepoint', 'subject'])
    _data.rename(columns={'description': 'crop_instance'}, inplace=True)

    # # def get_fdr_correct_pvals(_data, channels, crop_instances=['feature-full', 'center', 'periphery', 'gcs-full']):
    fig3_all_pvals[hues[0]] = {ch: get_fdr_correct_pvals(_data, channels=[ch], crop_instances=hues)[ch] for ch in ['Iz', 'Pz']}

In [ ]:
from scipy.stats import wilcoxon

def plot_max_timepoints(ax, channel, is_sig, y=0.3):
    center_timepoints = t_data.loc[t_data[(t_data['channel'] == channel) & (t_data['timepoint'] > 0) & (t_data['description'] == 'center | periphery')].groupby('subject').value.idxmax()].timepoint.values #  & (t_data['timepoint'] < 0.2)
    periphery_timepoints = t_data.loc[t_data[(t_data['channel'] == channel) & (t_data['timepoint'] > 0) & (t_data['description'] == 'periphery | center')].groupby('subject').value.idxmax()].timepoint.values #  & (t_data['timepoint'] < 0.2)

    # y = 0.3
    parts = ax.violinplot(center_timepoints, positions=[y], vert=False, widths=0.15, showmedians=True, showextrema=False, side='high')
    parts['cmedians'].set_color(colormaps['center'])
    for p in parts['bodies']:
        p.set_facecolor(colormaps['center'])
        p.set

    for _x in center_timepoints:
        ax.text(_x, y+0.01, '*', fontsize=30, color=colormaps['center'], ha='center', va='center', alpha=0.5)

    parts = ax.violinplot(periphery_timepoints, positions=[y], vert=False, widths=0.15, showmedians=True, showextrema=False, side='high')
    parts['cmedians'].set_color(tab20c.colors[0])
    for p in parts['bodies']:
        p.set_facecolor(tab20c.colors[0])

    for _x in periphery_timepoints:
        ax.text(_x, y, '*', fontsize=30, color=tab20c.colors[0], ha='center', va='center', alpha=0.5)

    stat = wilcoxon(center_timepoints, periphery_timepoints)
    is_sig = stat.pvalue < 0.05
    
    # draw a horizontal line between the two medians of the violins with small vertical extensions
    y += 0.09
    ax.plot([np.median(center_timepoints), np.median(periphery_timepoints)], [y, y], color='black', linewidth=2)
    ax.plot([np.median(center_timepoints), np.median(center_timepoints)], [y-0.0075, y+0.0075], color='black', linewidth=2)
    ax.plot([np.median(periphery_timepoints), np.median(periphery_timepoints)], [y-0.0075, y+0.0075], color='black', linewidth=2)
    ax.text((np.median(center_timepoints) + np.median(periphery_timepoints))/2, y+0.01, '*' if is_sig else 'NS', fontsize=30 if is_sig else 18, color='black', ha='center', va='center')

# fig, ax = plt.subplots(1, 1, figsize=(8, 8))
# plot_max_timepoints(ax, channel='Pz', is_sig=True, y=0.3)
# ax.set_xlim(-0.1, 0.4)
# ax.set_ylim(0, 0.5)
# plt.show()

In [ ]:
#### Plot #######
colormaps = {
    'feature-full': tab10.colors[5],
    'gcs-full': tab10.colors[0],
    'center': tab10.colors[1],
    'periphery': tab10.colors[7],
}

description_palette = {
                    'center | periphery': colormaps['center'], 
                    'periphery | center': tab20c.colors[0], # colormaps['periphery'],
                    'feature | gcs': colormaps['feature-full'],
                    'gcs | feature': 'black',
                    'center | gcs': tab20c.colors[4],
                    'gcs | center': 'black', #colormaps['gcs-full'],
                    'periphery | gcs': tab20c.colors[1],
                    'gcs | periphery': 'black' # tab20c.colors[9],
                    }

channels = [x for x in channel_names if x not in ['I1', 'I2', 'F5', 'F6', 'above', 'below', 'left', 'right']]


f = sns.relplot(kind='line', data=fig3_data, x='timepoint', y='value', hue='description', col='row', row='channel',  # , '5.0%', '20.0%'
                linewidth=6, 
                col_order=['center | periphery', 'center | gcs', 'periphery | gcs', 'gcs | feature'],
                hue_order = [
                    'center | periphery',
                    'periphery | center',
                    'center | gcs',
                    'gcs | center',
                    'periphery | gcs',
                    'gcs | periphery',
                    'feature | gcs',
                    'gcs | feature',
                ],
                palette=description_palette)
f.set_titles('')
f.set_axis_labels('Time (s)', 'Partial Correlation (r)', fontsize=30)



handles, lables = [], []
for _ax in f.axes.flatten():
    _ax.vlines(0.0, linestyles='dashed', ymin=0.0, ymax=0.5, colors='gray', alpha=0.5)
    _ax.hlines(0.0, linestyles='dashed', xmin=-0.1, xmax=0.4, colors='gray', alpha=0.5)

    _handles, _labels = _ax.get_legend_handles_labels()
    for handle, label in zip(_handles, _labels):
        label = label.replace('gcs', 'GCS').replace('feature', 'Full').replace('periphery', 'Periphery').replace('center', 'Center')
        if label not in lables:
            handles.append(handle)
            lables.append(label)

for ax_index in range(len(f.axes[0].flatten())):
    legend = f.axes.flatten()[ax_index].legend([handles[2*ax_index], handles[2*ax_index+1]], [lables[2*ax_index], lables[2*ax_index+1]], title='', loc='upper left', bbox_to_anchor=(0.05, 1.3), ncol=1, frameon=False, prop={'size': 28})

    for legobj in legend.legend_handles:
    # for legobj in legend.legendHandles:
        legobj.set_linewidth(5.0)
f._legend.remove()


_ax0 = f.axes.flatten()[0].inset_axes([0.8, 0.5, 0.2, 0.2])
_ax0.set_aspect('equal', anchor="C")
_ax0.axis('off')
_ax1 = f.axes.flatten()[4].inset_axes([0.8, 0.5, 0.2, 0.2])
_ax1.set_aspect('equal', anchor="C")
_ax1.axis('off')

plot_max_timepoints(f.axes.flatten()[0], channel='Iz', is_sig=True, y=0.53)
plot_max_timepoints(f.axes.flatten()[4], channel='Pz', is_sig=True, y=0.53)


inset_axes = [_ax0, _ax1]
for ax_index, ch in enumerate(['Iz', 'Pz']):
    info, tvalue_channel_names = make_custom_info(channels)
    mask_params = dict(marker='o', markerfacecolor='k', markeredgecolor='k',
            linewidth=0, markersize=10)
    channel_mask = np.array([x in [ch] for x in channels])

    _ax = plot_topomap(data=np.zeros((len(channels), )), pos=info, axes=inset_axes[ax_index], mask_params=mask_params, mask=channel_mask, sensors=False, show=False)
    inset_axes[ax_index].set_title(ch, fontweight='bold', fontsize=24)

    for col_index, hue in enumerate(['center | periphery', 'center | gcs', 'periphery | gcs', 'gcs | feature']):
        for timepoint in fig3_data.timepoint.unique():
            for cond_index, desc in enumerate(fig3_all_pvals[hue][ch].keys()):
                pval = fig3_all_pvals[hue][ch][desc][timepoint]
                if pval < 0.01:
                    f.axes[ax_index, col_index].plot(timepoint, -0.12-cond_index*0.02, '*', color=description_palette[desc])
            # f.axes[ax_index, col_index].

f.axes.flatten()[0].text(-0.15, 1.1, 'A', transform=f.axes.flatten()[0].transAxes, 
            size=48, weight='bold', clip_on=False)
f.axes.flatten()[1].text(-0.15, 1.1, 'B', transform=f.axes.flatten()[1].transAxes, 
            size=48, weight='bold', clip_on=False)
f.axes.flatten()[2].text(-0.15, 1.1, 'C', transform=f.axes.flatten()[2].transAxes, 
            size=48, weight='bold', clip_on=False)
f.axes.flatten()[3].text(-0.15, 1.1, 'D', transform=f.axes.flatten()[3].transAxes, 
            size=48, weight='bold', clip_on=False)

f.fig.subplots_adjust(wspace=0.2)
f.fig.set_size_inches(35, 12)

f.fig.savefig(os.path.join(figure_dir, f'Fig-3_partials_fig3.pdf'), dpi=300., bbox_inches='tight')

plt.show()

# Figure 4

In [ ]:
df = pd.read_parquet('../additional_results/alexnet_imagenet_additional_encoding_results_center-peri.parquet')

In [ ]:
peri_conds = ['peri1', 'peri2', 'peri3']
center_conds = ['center1', 'center2', 'center3']

In [ ]:
fig4_data = df[(df['channel'].isin(visual_channel_indices_res)) & (df['metric'] == 'test_corr_train') & (df['crop_instance'].str.contains('full'))].copy()
fig4_data.crop_instance = fig4_data.crop_instance.str.replace('gcs-full', 'GCS').str.replace('feature-full', 'Full')

fig4_data['size'] = fig4_data['exp_condition'].apply(lambda x: x[-1].replace('1', 'Small').replace('2', 'Medium').replace('3', 'Large'))
fig4_data['exp_condition'] = fig4_data.apply(lambda x: (x['exp_condition'][:-1].replace('center', 'Center').replace('peri', 'Periphery') + '-' + x['size'] if 'center' in x['exp_condition'] or 'peri' in x['exp_condition'] else x['exp_condition']), axis=1)

fig4_data.exp_condition = fig4_data.exp_condition.astype('category')
fig4_data.exp_condition = fig4_data.exp_condition.cat.remove_unused_categories()

In [ ]:
def plot_periphery_example_images(axes, n:int=3, titles=['Periphery', 'Periphery', 'Periphery']):

    images = {}
    for index in range(n):
        img_path = f'../examples_images/periphery/84c6eed4c48e78f0_index-{index}.png'
        img = Image.open(img_path)
        images[index] = img

    for index in range(n):
        axes[index].imshow(images[index])
        _size = images[index].size

        axes[index].axis('off')
        # axes[index].set_title(['Small', 'Medium', 'Large'][index])
        axes[index].set_title(titles[index])

def plot_center_example_images(axes, n:int=3, titles=['Center', 'Center', 'Center']):

    images = {}
    for index in range(n):
        img_path = f'../examples_images/center/84c6eed4c48e78f0_index-{index}.png'
        img = Image.open(img_path)
        images[index] = img

    for index in range(n):
        axes[index].imshow(images[index])
        _size = images[index].size

        axes[index].axis('off')
        # axes[index].set_title(['Small', 'Medium', 'Large'][index])
        axes[index].set_title(titles[index])


In [ ]:

# define an object that will be used by the legend
class MulticolorPatch(object):
    def __init__(self, colors, hatch=None, alpha=1.0):
        self.colors = colors
        self.hatch=hatch
        self.alpha=alpha
        
# define a handler for the MulticolorPatch object
class MulticolorPatchHandler(object):
    def legend_artist(self, legend, orig_handle, fontsize, handlebox):
        width, height = handlebox.width, handlebox.height
        patches = []
        for i, c in enumerate(orig_handle.colors):
            # print(orig_handle.hatch)
            patches.append(plt.Rectangle([width/len(orig_handle.colors) * i - handlebox.xdescent, 
                                          -handlebox.ydescent],
                           width / len(orig_handle.colors),
                           height,
                           facecolor=c, 
                           edgecolor='none'))

        patch = PatchCollection(patches, match_original=False, facecolor=[c for c in orig_handle.colors], hatch=orig_handle.hatch, alpha=orig_handle.alpha)
        handlebox.add_artist(patch)
        return patch

In [ ]:
_data = fig4_data[(fig4_data['channel'].isin([channel_names_res.index(ch) for ch in ['Iz', 'Pz']]))].copy() #  & (fig4_data['timepoint'] > 0.0)
# print(_data.channel.unique())
_data['channel'] = _data['channel'].astype('category').cat.reorder_categories([channel_names_res.index(ch) for ch in ['Iz', 'Pz']])
exp_condition_order = ['Center-Small', 'Center-Medium', 'Center-Large', 'Periphery-Small', 'Periphery-Medium', 'Periphery-Large']
_data['exp_condition'] = _data['exp_condition'].astype('category').cat.reorder_categories(exp_condition_order)

_data = _data[_data['crop_instance'] == 'Full'].copy()
_data['crop_instance'] = _data['crop_instance'].astype('category').cat.reorder_categories(['Full'])
_data = _data.sort_values(['exp_condition', 'crop_condition', 'channel', 'timepoint', 'subject'])

corrected_pvals = {exp_condition: {channel_names_res.index(ch): get_fdr_correct_pvals(_data[(_data['exp_condition'] == exp_condition)], channels=[channel_names_res.index(ch)], crop_instances=['Full'])[channel_names_res.index(ch)] for ch in ['Iz', 'Pz']} for exp_condition in exp_condition_order}

In [ ]:
def plot_centerperi_max_timepoints(df, ax, size, y=0.3):
    center_timepoints = df.loc[df[(df['crop_instance'] == 'Full') & (df['exp_condition'] == f'Center-{size}') & (df['timepoint'] > 0)].groupby('subject').value.idxmax()].timepoint.values #  & (df['timepoint'] < 0.2)
    periphery_timepoints = df.loc[df[(df['crop_instance'] == 'Full') & (df['exp_condition'] == f'Periphery-{size}') & (df['timepoint'] > 0)].groupby('subject').value.idxmax()].timepoint.values #  & (df['timepoint'] < 0.2)

    # y = 0.3
    parts = ax.violinplot(center_timepoints, positions=[y], vert=False, widths=0.15, showmedians=True, showextrema=False, side='high')
    parts['cmedians'].set_color(tab20c.colors[4])
    for p in parts['bodies']:
        p.set_facecolor(tab20c.colors[4])
        p.set

    for _x in center_timepoints:
        ax.text(_x, y+0.01, '*', fontsize=30, color=tab20c.colors[4], ha='center', va='center', alpha=0.5)

    parts = ax.violinplot(periphery_timepoints, positions=[y], vert=False, widths=0.15, showmedians=True, showextrema=False, side='high')
    parts['cmedians'].set_color(tab20c.colors[0])
    for p in parts['bodies']:
        p.set_facecolor(tab20c.colors[0])

    for _x in periphery_timepoints:
        ax.text(_x, y, '*', fontsize=30, color=tab20c.colors[0], ha='center', va='center', alpha=0.5)

    stat = wilcoxon(center_timepoints, periphery_timepoints)
    is_sig = stat.pvalue < 0.05

    # draw a horizontal line between the two medians of the violins with small vertical extensions
    y += 0.09
    ax.plot([np.median(center_timepoints), np.median(periphery_timepoints)], [y, y], color='black', linewidth=2)
    ax.plot([np.median(center_timepoints), np.median(center_timepoints)], [y-0.0075, y+0.0075], color='black', linewidth=2)
    ax.plot([np.median(periphery_timepoints), np.median(periphery_timepoints)], [y-0.0075, y+0.0075], color='black', linewidth=2)
    ax.text((np.median(center_timepoints) + np.median(periphery_timepoints))/2, y+0.01, '*' if is_sig else '', fontsize=30 if is_sig else 18, color='black', ha='center', va='center')


In [ ]:
# ACTUAL FIGURE CODE
fig = plt.figure(layout='constrained', figsize=(28, 26))

include_temporal_differences = True

subfigs = fig.subfigures(3, 1, height_ratios=[2, 4, 3])

extents = ['Small', 'Medium', 'Large']
images_subfigs = subfigs[0].subfigures(1, 3, wspace=0.1)
for i in range(3):
    images_subfigs[i].suptitle(extents[i], fontsize=36, y=0.93, fontweight='bold')
image_ax = np.array([images_subfigs[i].subplots(1,2) for i in range(3)]).flatten()

lineax = subfigs[1].subplots(2, 3, sharey=True, sharex=True) # 'row'

# lineax = ax[2, :]

_ax0 = lineax[0, 0].inset_axes([0.8, 0.3, 0.15, 0.15], clip_on=False, in_layout=False)
_ax0.set_aspect('equal', anchor="C")
_ax0.axis('off')
_ax1 = lineax[1, 0].inset_axes([0.8, 0.3, 0.15, 0.15], clip_on=False, in_layout=False)
_ax1.set_aspect('equal', anchor="C")
_ax1.axis('off')

inset_axes = [_ax0, _ax1]

# make_center_peri_boxplot(box_ax)
# make_center_peri_swarmplot(swarm_ax, titles=False)
plot_center_example_images(image_ax[::2].flatten())
plot_periphery_example_images(image_ax[1::2].flatten())

colormaps = {
    'Center-Small': tab20c.colors[4],
    'Center-Medium': tab20c.colors[4],
    'Center-Large': tab20c.colors[4],
    'Periphery-Small': tab20c.colors[0],
    'Periphery-Medium': tab20c.colors[0],
    'Periphery-Large': tab20c.colors[0],
}

plot_channels = [x for x in channel_names_res if x not in ['left', 'right', 'above', 'below', 'F5', 'F6', 'I1', 'I2', 'EXG1', 'EXG2', 'EXG3', 'EXG4']]

for row_index, ch in enumerate(['Iz', 'Pz']):
    
    info, tvalue_channel_names = make_custom_info(plot_channels)
    mask_params = dict(marker='o', markerfacecolor='k', markeredgecolor='k',
            linewidth=0, markersize=10)
    channel_mask = np.array([x in [ch] for x in plot_channels])

    _ax = plot_topomap(data=np.zeros((len(plot_channels), )), pos=info, axes=inset_axes[row_index], mask_params=mask_params, mask=channel_mask, sensors=False, show=False)
    inset_axes[row_index].set_title(ch, fontweight='bold', fontsize=28)
    
    for ax_index, size in enumerate(extents):
        _plot_data = fig4_data[(fig4_data['size'] == size) & (fig4_data['channel'].isin([channel_names_res.index(ch)]))].groupby(['timepoint', 'exp_condition', 'crop_instance', 'subject']).value.mean().reset_index().sort_values(['exp_condition', 'crop_instance', 'timepoint', 'subject'])
        g = sns.lineplot(data=_plot_data, 
                        x='timepoint', y='value', hue='exp_condition', hue_order=[x for x in colormaps.keys() if size in x], 
                        style='crop_instance', ax=lineax[row_index, ax_index], linewidth=3, palette=colormaps, legend='auto' if (ax_index == 1 and row_index == 0) else None)

        if include_temporal_differences:
            # center_t_max = _plot_data[(_plot_data['exp_condition'] == f'Center-{size}') & (_plot_data['crop_instance'] == 'Full')].groupby('timepoint').value.mean().idxmax()
            # periphery_t_max = _plot_data[(_plot_data['exp_condition'] == f'Periphery-{size}') & (_plot_data['crop_instance'] == 'Full')].groupby('timepoint').value.mean().idxmax()
            
            # lineax[row_index, ax_index].vlines(center_t_max, ymin=-0.1, ymax=0.6, color=colormaps[f'Center-{size}'], linestyle='--', linewidth=2)        
            # lineax[row_index, ax_index].vlines(periphery_t_max, ymin=-0.1, ymax=0.6, color=colormaps[f'Periphery-{size}'], linestyle='--', linewidth=2)
            plot_centerperi_max_timepoints(df=_plot_data, ax=lineax[row_index, ax_index], size=size, y=0.6)

        if row_index == 1:
            lineax[row_index, ax_index].set_xlabel('Time (s)', fontsize=28)
        else:
            lineax[row_index, ax_index].set_xlabel('')

        for timepoint in range(513):
            for exp_index, exp in enumerate(['Center', 'Periphery']):
                # corrected_pvals['Center-Small'][27]['Full'][t[2]]
                if corrected_pvals[f'{exp}-{extents[ax_index]}'][channel_names_res.index(ch)]['Full'][t[timepoint]] < 0.05:
                    lineax[row_index, ax_index].plot(t[timepoint], -0.05 - exp_index * 0.01, '*', color=colormaps[f'{exp}-{size}'])
                # if corrected_pvals[exp_index*3 + ax_index, 0, row_index, timepoint]: # < 0.05:
                #     lineax[row_index, ax_index].plot(t[timepoint], -0.05 - exp_index * 0.01, '*', color=colormaps[f'{exp}-{size}'])

        if ax_index == 1 and row_index == 0:
            handles, labels = lineax[row_index, ax_index].get_legend_handles_labels()
            handles = [handles[i] for i in [1,2,4,5]]
            labels = [labels[i].replace('-Small', '').replace('-Medium', '').replace('-Large', '') for i in [1,2,4,5]]
            legend = lineax[row_index, ax_index].legend(handles=handles, labels=labels, title='', ncols=4, fontsize=24, bbox_to_anchor=(0.5, -0.25), loc='upper center')
            plt.setp(legend.get_title(),fontsize=26)
            for legobj in legend.legend_handles:
                legobj.set_linewidth(5.0)

        if ax_index == 0:
            lineax[row_index, ax_index].set_ylabel('Encoding Performance (r)', fontsize=28)
        else:
            lineax[row_index, ax_index].set_ylabel('')
        
        if row_index == 0:
            lineax[row_index, ax_index].set_title(size, fontsize=30, y=1.15) # y=0.95

for ax_index, _ax in enumerate(lineax.flatten()):
    _ax.axvline(0, color='gray', linestyle='--', zorder=-10)
    _ax.axhline(0, color='gray', linestyle='--', zorder=-10)


############################################ Bar plots
bar_ax = subfigs[2].subplots(1, 2, gridspec_kw={'width_ratios': [1, 1], 'height_ratios': [1]}, sharey='row')

pairs = {cond: [
    # [('Small', 'Full'), ('Medium', 'Full')],
    # [('Medium', 'Full'), ('Large', 'Full')],
    # [('Small', 'Full'), ('Large', 'Full')],
    # [('Small', 'GCS'), ('Medium', 'GCS')],
    # [('Medium', 'GCS'), ('Large', 'GCS')],
    # [('Small', 'GCS'), ('Large', 'GCS')],

    [('Small', 'Full'), ('Small', 'GCS')],
    [('Medium', 'Full'), ('Medium', 'GCS')],
    [('Large', 'Full'), ('Large', 'GCS')],
] for cond in ['Center', 'Periphery']}

fig4_barplot_data = fig4_data[(fig4_data['timepoint'] > 0)].copy()
for ax_index, cond in enumerate(['Center', 'Periphery']):
    _data_cond = fig4_barplot_data[fig4_barplot_data['exp_condition'].str.contains(cond)].copy()
    _data_cond.exp_condition = _data_cond.exp_condition.cat.remove_unused_categories()
    _data_cond.exp_condition = _data_cond.exp_condition.str.replace('Center-', '').str.replace('Periphery-', '')
    # _data_cond = _data_cond.groupby(['subject', 'exp_condition', 'crop_instance']).value.mean().reset_index()
    # _data_cond = _data_cond.groupby(['subject', 'channel', 'exp_condition', 'crop_instance']).value.apply(lambda x: auc(_data_cond.timepoint.unique(), x)).reset_index()
    _data_cond = _data_cond.groupby(['subject', 'exp_condition', 'crop_instance']).value.mean().reset_index()
    
    order = ['Small', 'Medium', 'Large']

    plotting_parameters = {
        'data':    _data_cond,
        'x':       'exp_condition',
        'y':       'value',
        'hue':     'crop_instance',
        'ax':      bar_ax[ax_index],
        'order':   order,
        # 'hide_non_significant': True,
    }
    # g = sns.barplot(data=_data_cond, x='exp_condition', y='value', hue='crop_instance', ax=ax[ax_index], order=order)#, palette=_cmap)
    g = sns.barplot(**plotting_parameters)
    # _ = sns.stripplot(**plotting_parameters)

    annotator = Annotator(pairs=pairs[cond], **plotting_parameters)
    # annotator.configure(hide_non_significant=True)
    # annotator.configure(test='Wilcoxon', comparisons_correction='Benjamini-Hochberg', hide_non_significant=True)
    annotator.configure(test='Wilcoxon', hide_non_significant=True)
    annotator.apply_and_annotate()
    # annotator.configure(comparisons_correction=None, hide_non_significant=True)
    # annotator.set_pvalues(fig4_formatted_pvals[cond])
    # annotator.annotate()

    # color bars differently
    for i, bar in enumerate(g.patches[:-2]): # exclude legend
        if i > 2:
            bar.set_hatch('//')
            i -= 3

        bar.set_color(colormaps[f"{cond}-{order[i]}"])
        bar.set_edgecolor('black')

    bar_ax[ax_index].set_xlabel('')
    if ax_index == 0:
        bar_ax[ax_index].set_ylabel('Average encoding performance (r)')
        # bar_ax[ax_index].set_ylabel('Average Area Under Curve (AUC)', fontsize=28)
    else:
        bar_ax[ax_index].set_ylabel('')

    bar_ax[ax_index].set_title(cond, fontsize=30, fontweight='bold') # cond
    g.get_legend().remove()

    new_h = [
                MulticolorPatch([tab20c.colors[4]]) if cond == 'Center' else MulticolorPatch([tab20c.colors[0]]),
                MulticolorPatch([tab20c.colors[4]], hatch='//') if cond == 'Center' else MulticolorPatch([tab20c.colors[0]], hatch='//'),
            ] # , hatch='//', alpha=0.8
    labels = ['Full', 'GCS']
    legend = g.axes.legend(handles=new_h, labels=labels, title='Encoding Model', ncol=2, fontsize=24, frameon=True, handler_map={MulticolorPatch: MulticolorPatchHandler()})      
    plt.setp(legend.get_title(),fontsize=26)
############################################

sns.despine()

image_ax[0].text(0.0, 1.2, 'A', color='black', size=48, weight='bold', transform=image_ax[0].transAxes, clip_on=False) # , in_layout=False)
lineax[0, 0].text(-0.2, 1.2, 'B', color='black', size=48, weight='bold', transform=lineax[0, 0].transAxes, clip_on=False) # , in_layout=False)
bar_ax[0].text(-0.2, 1.0, 'C', color='black', size=48, weight='bold', transform=bar_ax[0].transAxes, clip_on=False) # , in_layout=False)

for _ax in inset_axes:
    _ax.set_zorder(100)

fig.savefig(os.path.join(figure_dir, 'Fig-4_center_peri_conditions.pdf'), dpi=300, bbox_inches='tight')
plt.show()

# Figure 5 (Sizes)

In [ ]:
df_sizes = pd.read_parquet('../additional_results/alexnet_imagenet_additional_encoding_results_sizes.parquet')

In [ ]:
sizes_data = df_sizes[(df_sizes['channel'].isin(visual_channel_indices_res)) & (df_sizes['metric'] == 'test_corr_train')].copy()
sizes_data['exp_condition'] = sizes_data['exp_condition'].apply(lambda x: x[-1].replace('1', 'Small').replace('2', 'Medium').replace('3', 'Large'))
sizes_data['crop_instance'] = sizes_data['crop_instance'].apply(lambda x: x.replace('gcs-full', 'GCS').replace('feature-full', 'Full'))

In [ ]:
def plot_size_example_images(axes, n:int=3, increase_title_font:bool=False):

    images = {}
    for size in range(n):
        img_path = f'../examples_images/size/84c6eed4c48e78f0_index-{size}.png'
        img = Image.open(img_path)
        images[size] = img

    max_x = max([img.size[0] for img in images.values()])
    max_y = max([img.size[1] for img in images.values()])
    # max_x, max_y

    # fig, ax = plt.subplots(1,3, figsize=(15, 10))

    medium_rect = None

    for size in range(3):
        axes[size].imshow(images[size])
        _size = images[size].size

        if size == 0 or size == 1:
            axes[size].set_xlim(-max_x//2 + _size[0]//2, max_x//2 + _size[0]//2)
            axes[size].set_ylim(max_y//2 + _size[1]//2, -max_y//2 + _size[1]//2)

        if size == 1:
            medium_rect = patches.Rectangle((-_size[0]//4, -_size[1]//4), _size[0], _size[1], linewidth=1, edgecolor='k', linestyle='--', facecolor='none')
            axes[0].add_patch(medium_rect)

        axes[size].set_xticks([])
        axes[size].set_yticks([])
        if increase_title_font:
            axes[size].set_title(['Small', 'Medium', 'Large'][size], fontsize=30, fontweight='bold')
        else:
            axes[size].set_title(['Small', 'Medium', 'Large'][size])

    # plt.show()

In [ ]:
def plot_size_noise_ceilings(ax):
    for cond in ['size1', 'size2', 'size3']:
        label = cond.replace('size1', 'Small').replace('size2', 'Medium').replace('size3', 'Large')
        _mean = np.mean([noise_ceiling_res[cond][sub]['lower'][visual_channel_indices_res].mean(axis=0) for sub in [1, 2, 3, 4, 12, 15, 16, 17, 18, 19]], axis=0)
        _ste = np.std([noise_ceiling_res[cond][sub]['lower'][visual_channel_indices_res].mean(axis=0) for sub in [1, 2, 3, 4, 12, 15, 16, 17, 18, 19]], axis=0) / np.sqrt(10)

        ax.plot(t, _mean, label=label, linestyle='--', linewidth=4)
        ax.fill_between(t, _mean - _ste, _mean + _ste, alpha=0.2)
    ax.legend()
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Correlation (r)')
    ax.set_title('Noise Ceiling (lower bound)')

In [ ]:
sizes_data[((sizes_data['channel'] == channel_names_res.index(lineplot_channel)))].groupby(['exp_condition', 'crop_instance']).value.std()

In [ ]:
fig, ax = plt.subplots(4,6, figsize=(30,12), sharey=False, sharex=False, gridspec_kw={'width_ratios': [4, 4, 4, 0.5, 2, 2], 'height_ratios': [1, 0.2, 1, 1]})

for _ax in ax[1, :].flatten():
    _ax.axis('off')

# corrected_data['exp_condition'] = corrected_data['exp_condition'].str.replace('size1', 'Small').str.replace('size2', 'Medium').str.replace('size3', 'Large')
sizes = ['size1', 'size2', 'size3']

colormaps = {
    'Full': tab10.colors[5],
    'GCS': 'black',
}

lineplot_channel = 'Iz'

for ax_index, condition in enumerate(['Small', 'Medium', 'Large']):
    _data = sizes_data[((sizes_data['exp_condition'] == condition) & (sizes_data['channel'] == channel_names_res.index(lineplot_channel)))].copy().reset_index()
    
    f = sns.lineplot(data=_data, x='timepoint', y='value', hue='crop_instance', ax=ax[2, ax_index], linewidth=3, palette=colormaps, legend='full' if ax_index == 1 else False)
    ax[2, ax_index].set_title(condition)
    # ax[2, ax_index].set_xlabel('Time (s)')

    ax[2, ax_index].set_xlabel('')
    ax[2, ax_index].set_ylabel('')
    ax[2, ax_index].set_xticks([])

    if ax_index == 1:
        f.legend(title='', loc='upper left')
    else:
        f.legend().set_visible(False)
    
    if ax_index > 0:
        ax[2, ax_index].set_yticks([])

    ax[2, ax_index].axvline(0, color='gray', linestyle='--')
    ax[2, ax_index].axhline(0, color='gray', linestyle='--')

    average_data = sizes_data[((sizes_data['exp_condition'] == condition) & (sizes_data['channel'] == channel_names_res.index('Pz')))].copy().reset_index()

    f = sns.lineplot(data=average_data, x='timepoint', y='value', hue='crop_instance', ax=ax[3, ax_index], linewidth=3, palette=colormaps, legend=False)

    ax[3, ax_index].set_xlabel('')
    ax[3, ax_index].set_ylabel('')
    if ax_index == 1:
        ax[3, ax_index].set_xlabel('Time (s)')

    if ax_index > 0:
        ax[3, ax_index].set_yticks([])
    
    ax[3, ax_index].axvline(0, color='gray', linestyle='--')
    ax[3, ax_index].axhline(0, color='gray', linestyle='--')
    # ===========================================================================


# ============================================================================================== Iz
left, bottom, width, height = [0.11, 0.5, 0.075, 0.075]
inset_ax0 = fig.add_axes([left, bottom, width, height])

info, tvalue_channel_names = make_custom_info(visual_channel_names)
mask_params = dict(marker='o', markerfacecolor='k', markeredgecolor='k',
        linewidth=0, markersize=6)
channel_mask = np.array([x in [lineplot_channel] for x in visual_channel_names])

_ax = plot_topomap(data=np.zeros((len(visual_channel_names), )), pos=info, axes=inset_ax0, mask_params=mask_params, mask=channel_mask, sensors=False, show=False)
inset_ax0.set_title(lineplot_channel, fontweight='bold', y=0.2)
# ==============================================================================================

# ============================================================================================== Pz
left, bottom, width, height = [0.11, 0.2, 0.075, 0.075]
inset_ax1 = fig.add_axes([left, bottom, width, height])

info, tvalue_channel_names = make_custom_info(visual_channel_names)
mask_params = dict(marker='o', markerfacecolor='k', markeredgecolor='k',
        linewidth=0, markersize=6)
channel_mask = np.array([x in ['Pz'] for x in visual_channel_names])


_ax = plot_topomap(data=np.zeros((len(visual_channel_names), )), pos=info, axes=inset_ax1, mask_params=mask_params, mask=channel_mask, sensors=False, show=False)
inset_ax1.set_title('Pz', fontweight='bold', y=0.3)
# ==============================================================================================


# ==============================================================================================
plot_size_example_images(ax[0, :-3], 3)

gs = ax[0, -2].get_gridspec()
for _ax in ax[0, -2:]:
    _ax.remove()
_ax = fig.add_subplot(gs[0, -2:])

plot_size_noise_ceilings(_ax)
_ax.axvline(0, color='gray', linestyle='--')
_ax.axhline(0, color='gray', linestyle='--')


# ================================================
pairs = [
    [('Small', 'Full'), ('Small', 'GCS')],
    [('Medium', 'Full'), ('Medium', 'GCS')],
    [('Large', 'Full'), ('Large', 'GCS')],
]

plotting_parameters = {
    'data':    sizes_data.groupby(['subject', 'crop_instance', 'exp_condition']).mean(numeric_only=True).reset_index(),
    'x':       'exp_condition',
    'y':       'value',
    'hue':     'crop_instance',
    'palette':  colormaps,
    'order':    ['Small', 'Medium', 'Large'],
    'hue_order': ['Full', 'GCS'],
    # 'ax': ax,
}

f = sns.barplot(ax=ax[2, -1], **plotting_parameters)

gs = ax[2, -2].get_gridspec()
for _ax in ax[2:, -2:].flatten():
    _ax.remove()

_axbox = fig.add_subplot(gs[-2:, -2:])

f = sns.barplot(ax=_axbox, **plotting_parameters)
size_pairs = [
    (('Small', 'Full'), ('Small', 'GCS')),
    (('Medium', 'Full'), ('Medium', 'GCS')),
    (('Large', 'Full'), ('Large', 'GCS')),
]

annotator = Annotator(pairs=size_pairs, ax=_axbox, **plotting_parameters)
annotator.configure(test='Wilcoxon', hide_non_significant=True, verbose=True)
annotator.apply_and_annotate()

f.set_xlabel('')
f.set_ylabel('Average encoding performance (r)')

_axbox.legend(title='', loc='upper left', ncol=1, bbox_to_anchor=(0.05, 0.8)) # (0.6, 1.2)


sns.despine()
ax[0, -3].axis('off')
ax[2, -3].axis('off')
ax[3, -3].axis('off')

for _ax in ax[0, :-3]:
    _ax.spines[['top', 'right', 'left', 'bottom']].set_visible(True)
    _ax.set_xticks([])
    _ax.set_yticks([])

fig.text(-0.35, 1.0, 'A', size=48, weight='bold', clip_on=False, transform=ax[0, 0].transAxes)
fig.text(3.8, 1.0, 'B', size=48, weight='bold', clip_on=False, transform=ax[0, 0].transAxes)
fig.text(-0.35, -0.4, 'C', size=48, weight='bold', clip_on=False, transform=ax[0, 0].transAxes)
fig.text(3.8, -0.4, 'D', size=48, weight='bold', clip_on=False, transform=ax[0, 0].transAxes)

fig.text(-0.15, 1.1, 'Encoding performance (r)', va='center', ha='center', rotation='vertical', wrap=True, transform=ax[-1, 0].transAxes, fontsize=24)

fig.subplots_adjust(hspace=0.2)

# fig.savefig(os.path.join(figure_dir, 'Fig-5_size_conditions_fig5.pdf'), dpi=300, bbox_inches='tight')  

plt.show()

# Figure 6 (Importance maps)

In [ ]:
with open('../results/rotations_per_channel.pkl', 'rb') as f:
    sub_rotations = pickle.load(f)

In [ ]:
time_eccentricity_curves = [{sub: [sub_rotations[sub][channel, :, timepoint] - np.mean(sub_rotations[sub][channel, :, :100], axis=1) for channel in range(sub_rotations[sub].shape[0])] for sub in sub_rotations.keys()} for timepoint in range(n_timepoints)]

In [ ]:
# THIS CELL TAKES THE LONGEST (4 mins)

rows = []
cols = ['subject', 'channel_index', 'channel', 'timepoint_index', 'timepoint', 'beta', 'intercept']
time_betas = []
for timepoint in range(n_timepoints):
    betas = {sub: [] for sub in sub_rotations.keys()}
    for sub, curves in time_eccentricity_curves[timepoint].items():
        for ch_idx, curve in enumerate(curves):
            x = np.arange(len(curve)).reshape(-1, 1)
            lin_reg = LinearRegression()
            lin_reg.fit(x, curve)
            betas[sub].append(lin_reg.coef_[0])
            
            rows.append([sub, ch_idx, channel_names[ch_idx], timepoint, f'{t[timepoint]:.3f}', lin_reg.coef_[0], lin_reg.intercept_])

    time_betas.append(betas)

In [ ]:
rot_df = pd.DataFrame(rows, columns=cols)

In [ ]:
rot_df = rot_df[(rot_df['channel'].isin([x for x in channel_names if x not in ['F5', 'F6']]))]

In [ ]:
from mne.stats import permutation_t_test

# channel_order = rot_df.sort_values('channel').channel.unique()
channel_order = [x for x in channel_names if x not in ['F5', 'F6', 'I1', 'I2', 'above', 'below', 'left', 'right']]
rot_df['channel'] = pd.Categorical(rot_df['channel'], categories=channel_order, ordered=True)

intercepts = rot_df.sort_values(['subject', 'channel', 'timepoint']).intercept.values.reshape((rot_df.subject.nunique(), -1))
_, pvals_intercept, _ = permutation_t_test(intercepts)

# bring back into channelxtimepoint shape
pvals_intercept = pvals_intercept.reshape((len(channel_order), len(rot_df.timepoint_index.unique())))

betas = rot_df.sort_values(['subject', 'channel', 'timepoint']).beta.values.reshape((rot_df.subject.nunique(), -1))
_, pvals_beta, _ = permutation_t_test(betas)

# bring back into channelxtimepoint shape
pvals_beta = pvals_beta.reshape((len(channel_order), len(rot_df.timepoint_index.unique())))

In [ ]:
n_subs = rot_df.subject.nunique()
print(n_subs)

In [ ]:
plot_data = {
    'betas': [],
    'intercepts': [],
    'sig_inter_pvals': [],
    'sig_beta_pvals': [],
    'inter_pvals': [],
    'beta_pvals': []
}

In [ ]:
plot_data[f'sig_inter'] = (pvals_intercept < 0.05).T
plot_data[f'intercepts'] = (intercepts.mean(axis=0).reshape((len(channel_order), len(rot_df.timepoint_index.unique())))).T

plot_data[f'sig_beta'] = (pvals_beta < 0.05).T
plot_data[f'betas'] = (betas.mean(axis=0).reshape((len(channel_order), len(rot_df.timepoint_index.unique())))).T

In [ ]:
def make_topoplot_subfig_c(ax, channels, plot_timepoints = [190, 210, 250]):
    # channels = [x for x in rot_df['channel'].unique() if x not in ['F5', 'F6']]
    info, tvalue_channel_names = make_custom_info(channels)

    min_beta = rot_df[(rot_df['timepoint_index'].isin(plot_timepoints)) & (rot_df['channel'].isin(channels))].beta.min()
    # max_beta = rot_df[(rot_df['timepoint_index'].isin(plot_timepoints)) & (rot_df['channel'].isin(channels))].beta.max()
    max_inter = rot_df[(rot_df['timepoint_index'].isin(plot_timepoints)) & (rot_df['channel'].isin(channels))].intercept.max()



    for ax_index, timepoint in enumerate(plot_timepoints):
        for i, _ax in enumerate(ax[:, ax_index]):
            _ax.set_aspect('equal', anchor="NW")
            _ax.axis('off')
            if i == 0:
                _ax.set_title(f'{t[timepoint]*1000:.0f} ms')
        
        mask_params = dict(marker='o', markerfacecolor='g', markeredgecolor='g',
                linewidth=0, markersize=8)
        cmap = 'seismic'
        # print(ax_index)
        
        im = plot_topomap(data=plot_data['intercepts'][timepoint], pos=info, axes=ax[0, ax_index], mask_params=mask_params, mask=plot_data['sig_inter'][timepoint], sensors=True, show=False, vlim=(-max_inter, max_inter), cmap=cmap)
        fig.colorbar(im[0], ax=ax[0, ax_index], shrink=0.7) # label='Intercept', 

        im = plot_topomap(data=plot_data['betas'][timepoint], pos=info, axes=ax[1, ax_index], mask_params=mask_params, mask=plot_data['sig_beta'][timepoint], sensors=True, show=False, vlim=(min_beta, -min_beta), cmap=cmap)
        fig.colorbar(im[0], ax=ax[1, ax_index], shrink=0.7) # label='Beta', 


In [ ]:
def make_eccentricity_subfig_b(ax, rot_df, sub_time_ecc, timepoints, channel_names, sub_rotations, t, lims, do_channels=['Iz', 'Pz']):
    for ax_index, channel in enumerate(do_channels):
        sub_betas = {timepoint: [] for timepoint in timepoints}
        sub_intercepts = {timepoint: [] for timepoint in timepoints}
        sub_time_ecc = {timepoint: [] for timepoint in timepoints}
        sub_ecc = []

        for sub in rot_df.subject.unique():
            for timepoint in timepoints:
                sub_betas[timepoint].append(rot_df[(rot_df['timepoint_index'] == timepoint) & (rot_df['channel'] == channel) & (rot_df['subject'] == sub)]['beta'].values[0])
                sub_intercepts[timepoint].append(rot_df[(rot_df['timepoint_index'] == timepoint) & (rot_df['channel'] == channel) & (rot_df['subject'] == sub)]['intercept'].values[0])
                time_ecc = time_eccentricity_curves[timepoint][sub][channel_names.index(channel)]
                sub_time_ecc[timepoint].append(time_ecc)

            ecc = sub_rotations[sub][channel_names.index(channel), :, :]
            sub_ecc.append(ecc)


        # im = ax[ax_index, 0].imshow(sub_rotations[sub][channel_names.index(channel), :, :], aspect='auto', origin='lower', cmap='viridis', clim=(-0.1, 0.6))
        im = ax[ax_index, 0].imshow(np.mean(sub_ecc, axis=0), aspect='auto', origin='lower', cmap='viridis', clim=lims)
        plt.colorbar(im, ax=ax[ax_index, 0]).set_label(label='Correlation', size=18)
        if ax_index == 1:
            ax[ax_index, 0].set_xlabel('Time (ms)')
            ax[ax_index, 0].set_xticks([0, 100, 200, 300, 400, 500])
            ax[ax_index, 0].set_xticklabels([-100, 0, 100, 200, 300, 400])
        ax[ax_index, 0].set_ylabel('Eccentricity')
        # ax[ax_index, 0].set_title(f'Radial average', fontsize=26, fontweight='bold')

        for idx, timepoint in enumerate(timepoints):
            ax[ax_index, 0].axvline(x=timepoint, color='red', linestyle='--')

            # ax[ax_index, 1+idx].plot(time_eccentricity_curves[timepoint][sub][channel_names.index(channel)], label=channel)
            _mean = np.mean(sub_time_ecc[timepoint], axis=0)
            _std = np.std(sub_time_ecc[timepoint], axis=0)
            _ste = _std / np.sqrt(len(sub_time_ecc[timepoint]))
            ecc = range(len(_mean)) # Change this to reflect degrees?
            ax[ax_index, 1+idx].plot(ecc, _mean, label=f'Data: {channel} - {t[timepoint]*1000:.0f} ms', linestyle='dotted', linewidth=4) # channel
            ax[ax_index, 1+idx].fill_between(np.arange(len(_mean)), _mean - _ste, _mean + _ste, alpha=0.5)


            # ax[ax_index, 1+idx].plot(np.arange(len(sub_rotations[sub][channel_names.index(channel), :, :].mean(axis=1))) * beta + intercept, label=f'Linear fit: {beta:.3f}x + {intercept:.2f}')
            _mean_beta = np.mean(sub_betas[timepoint])
            _mean_intercept = np.mean(sub_intercepts[timepoint])
            _std_beta = np.std(sub_betas[timepoint])
            _std_intercept = np.std(sub_intercepts[timepoint])
            _ste_beta = _std_beta / np.sqrt(len(sub_betas[timepoint]))
            _ste_intercept = _std_intercept / np.sqrt(len(sub_intercepts[timepoint]))
            x = np.arange(len(sub_rotations[sub][channel_names.index(channel), :, :].mean(axis=1)))
            ax[ax_index, 1+idx].plot(x * _mean_beta + _mean_intercept, label=f'Linear fit:\n{_mean_beta:.3f}x + {_mean_intercept:.2f}', color='k', linestyle='solid', linewidth=2)
            
            # if ax_index == 1:
            #     ax[ax_index, 1+idx].set_xlabel('Eccentricity', fontsize=28)
            # if idx == 0:
            #     ax[ax_index, 1+idx].set_ylabel('Correlation', fontsize=28)
            ax[ax_index, 1+idx].legend()
            ax[ax_index, 1+idx].set_ylim(lims)
            ax[ax_index, 1+idx].set_title(f'{channel} - {t[timepoint]*1000:.0f} ms')
            ax[ax_index, 1+idx].set_xlabel('Eccentricity')

            if idx == 0:
                ax[ax_index, 1+idx].set_ylabel('Correlation')

            ax[ax_index, 1+idx].spines[['top', 'right']].set_visible(False)

    ax[0, 2].set_yticklabels([])
    ax[1, 2].set_yticklabels([])

# plt.show()

In [ ]:
with open('../results/mean_contribution.pkl', 'rb') as f:
    mean_contribution = pickle.load(f)
    clims = mean_contribution['clims']
    mean_contribution = mean_contribution['mean_contribution']

In [ ]:
fig = plt.figure(figsize=(32, 16), layout='constrained')

subfigs = fig.subfigures(1, 2, width_ratios=[1, 2], wspace=0.05)

ax = subfigs[0].subplots(4, 2)

timepoints = [200, 250]
# clims = np.array([sub_contributions[5][channel_names.index(ch)][timepoint] for ch in ['Iz', 'Oz', 'Pz', 'P8'] for timepoint in timepoints]).flatten()
# clims = min(clims), max(clims)

for row_index, ch in enumerate(['Iz', 'Oz', 'Pz', 'P8']):
    for col_index, timepoint in enumerate(timepoints):
        im = ax[row_index, col_index].imshow(mean_contribution[col_index+1, row_index], cmap='viridis', clim=clims)
        fig.colorbar(im, ax=ax[row_index, col_index], shrink=0.6, label='Correlation')
        ax[row_index, col_index].axis('off')
        ax[row_index, col_index].set_title(f'{ch} - {t[timepoint]*1000:.0f} ms')


row_subfigs = subfigs[1].subfigures(2, 1, hspace=0.2)

ax = row_subfigs[0].subplots(2, 3, sharex='col', sharey='col', gridspec_kw={'width_ratios': [1, 1, 1]})
timepoints = [200, 250]
lims = (0.0, 0.45)
make_eccentricity_subfig_b(ax, rot_df, time_eccentricity_curves, timepoints, channel_names, sub_rotations, t, lims, do_channels=['Iz', 'Pz'])

plot_timepoints = [185, 200, 250]
ax = row_subfigs[1].subplots(2, len(plot_timepoints)) # , gridspec_kw={'wspace': 0.05}

channels = [x for x in channel_names if x not in ['F5', 'F6', 'above', 'below', 'left', 'right']]
make_topoplot_subfig_c(ax, channels, plot_timepoints=plot_timepoints)

fig.text(0.0, 1.02, 'A', color='black', size=48, weight='bold', clip_on=False) # , in_layout=False)
fig.text(0.10, 1.02, 'Contribution maps', color='black', size=36, weight='bold', clip_on=False) # , in_layout=False)

fig.text(1/3, 1.02, 'B', color='black', size=48, weight='bold', clip_on=False) # , in_layout=False)
fig.text(0.57, 1.02, 'Radial averages', color='black', size=36, weight='bold', clip_on=False) # , in_layout=False)

fig.text(1/3, 0.5, 'C', color='black', size=48, weight='bold', clip_on=False) # , in_layout=False)
fig.text(0.57, 0.5, 'Intercepts and betas', color='black', size=36, weight='bold', clip_on=False) # , in_layout=False)

fig.text(1/3+0.05, 0.275, 'Intercept', color='black', size=28, rotation=90, weight='bold', clip_on=False) # , in_layout=False)
fig.text(1/3+0.05, 0.075, 'Beta', color='black', size=28, rotation=90, weight='bold', clip_on=False) # , in_layout=False)

# plt.savefig(os.path.join(figure_dir, 'Fig-6_importance_maps_fig6.pdf'), dpi=300, bbox_inches='tight')

plt.show()

# Fig. 7 - Spatially optimized encoding performance

In [ ]:
opt_df = pd.read_parquet('../results/alexnet_imagenet_encoding_results_sampling.parquet')

In [ ]:
cols = opt_df.columns.tolist()

In [ ]:
comb_data = pd.concat([
    opt_df,
    _main_data[(_main_data['crop_condition'].isin(['feature', 'gcs'])) & (_main_data['subject'].isin(opt_df.subject.unique())) & (_main_data['timepoint'].isin(opt_df.timepoint.unique()))].rename(columns={'crop_condition': 'spatial_model'}, inplace=False)[cols]
])

In [ ]:
avg_rows = []
avg_cols = ['subject', 'spatial_model', 'channel', 'metric', 'value']

for sub in comb_data.subject.unique():
    for channel in visual_channel_names:
        for spatial_model in comb_data.spatial_model.unique():
            _df = comb_data[(comb_data['subject'] == sub) & (comb_data['channel'] == channel) & (comb_data['spatial_model'] == spatial_model)].sort_values('timepoint')
            avg_val = _df.value.mean()
            avg_rows.append([sub, spatial_model, channel, 'test_corr', avg_val])

avg_df = pd.DataFrame(avg_rows, columns=avg_cols)

In [ ]:
from statannotations.Annotator import Annotator

In [ ]:
spo = avg_df[avg_df['spatial_model'] == 'spatially-optimized']
gcs = avg_df[avg_df['spatial_model'] == 'gcs']

diff_df = spo.copy()
diff_df['value'] = spo.value.values - gcs.value.values

In [ ]:
from scipy.stats import wilcoxon

In [ ]:
# channel_pvals = diff_df.groupby('channel').apply(lambda x: ttest_1samp(x.value.values, popmean=0)).to_dict()
channel_pvals = diff_df.groupby('channel').apply(lambda x: wilcoxon(x.value.values)).to_dict()
channel_pvals = [channel_pvals[channel] if channel in channel_pvals.keys() else None for channel in [x for x in channel_names if x not in ['left', 'right', 'above', 'below', 'F5', 'F6']]]

In [ ]:
median_vals = diff_df.groupby('channel').value.median().to_dict()
median_vals = [median_vals[channel] if channel in median_vals.keys() else None for channel in [x for x in channel_names if x not in ['left', 'right', 'above', 'below', 'F5', 'F6']]]

In [ ]:
# create a cmap that goes from one specified color to another
start_color = 'black'
end_color = tab10.colors[2]
topo_cmap = LinearSegmentedColormap.from_list('custom_cmap', [start_color, 'white', end_color])

In [ ]:
# fig, ax = plt.subplots(1, 3, figsize=(28, 6), gridspec_kw={'wspace': 0.8, 'width_ratios': [1, 1, 3]})
fig = plt.figure(figsize=(28, 7), layout='constrained')
subfigs = fig.subfigures(1, 2, width_ratios=[3, 2], wspace=0.05)

first_ax = subfigs[0].subplots(1, 2)
second_ax = subfigs[1].subplots(1, 1)

ax = np.array([first_ax[0], first_ax[1], second_ax])

colormaps = {
    'feature': tab10.colors[5],
    'gcs': '0.3', #tab10.colors[0],
    'spatially-optimized': tab10.colors[2],
}

plotting_parameters = {
    'data':    avg_df.groupby(['subject', 'spatial_model']).value.mean().reset_index(),
    'x':       'spatial_model',
    'hue':     'spatial_model',
    'y':       'value',
    'order':    ['feature', 'gcs', 'spatially-optimized'],
    'palette': colormaps,
}

f = sns.barplot(ax=ax[0], **plotting_parameters, legend=False)
f = sns.stripplot(ax=ax[0], **plotting_parameters, edgecolor='k', linewidth=2, legend=False)

pairs = [
    ('spatially-optimized', 'gcs'),
    ('spatially-optimized', 'feature'),
    ('feature', 'gcs'),
]

annotator = Annotator(pairs=pairs, ax=ax[0], **plotting_parameters, verbose=True)
annotator.configure(test='Wilcoxon', hide_non_significant=True, pvalue_thresholds=[[1e-3, '***'], [1e-2, '**'], [1e-1, '*']])
# annotator.configure(test='Wilcoxon', hide_non_significant=True)
annotator.apply_and_annotate()

ax[0].set_xlabel('Spatial model')
ax[0].set_ylabel('Encoding performance (r)')

ax[0].spines[['top', 'right']].set_visible(False)
ax[0].set_xticks([0, 1, 2], ['Full', 'GCS', 'Sampling'])
# set maximally 4 yticks
from matplotlib.ticker import MaxNLocator
ax[0].yaxis.set_major_locator(MaxNLocator(nbins=4))

info, _ = make_custom_info([x for x in channel_names if x not in ['left', 'right', 'above', 'below', 'F5', 'F6']])
mask_params = dict(marker='o', markerfacecolor=tab10.colors[2], markeredgecolor='k',
        linewidth=0, markersize=6)
# channel_mask = np.array([x.pvalue < 0.05 and x.statistic > 0 if x is not None else False for i, x in enumerate(channel_pvals)])
# im, _ = plot_topomap(data=[x.statistic if x is not None else 0 for x in channel_pvals], pos=info, mask_params=mask_params, mask=channel_mask, sensors=True, show=False, axes=ax[1], cmap=topo_cmap)
channel_mask = np.array([x.pvalue < 0.05 and median_vals[i] > 0 if x is not None else False for i, x in enumerate(channel_pvals)])
im, _ = plot_topomap(data=[x if x is not None else 0 for x in median_vals], pos=info, mask_params=mask_params, mask=channel_mask, sensors=True, show=False, axes=ax[1], cmap=topo_cmap)

mask_params = dict(marker='o', markerfacecolor='k', markeredgecolor='k',
        linewidth=0, markersize=6)
# channel_mask = np.array([x.pvalue < 0.05 and x.statistic < 0 if x is not None else False for i, x in enumerate(channel_pvals)])
# im, _ = plot_topomap(data=[x.statistic if x is not None else 0 for  x in channel_pvals], pos=info, mask_params=mask_params, mask=channel_mask, sensors=True, show=False, axes=ax[1], cmap=topo_cmap)
channel_mask = np.array([x.pvalue < 0.05 and median_vals[i] > 0 if x is not None else False for i, x in enumerate(channel_pvals)])
im, _ = plot_topomap(data=[x if x is not None else 0 for x in median_vals], pos=info, mask_params=mask_params, mask=channel_mask, sensors=True, show=False, axes=ax[1], cmap=topo_cmap)

cbar = plt.colorbar(im, ax=ax[1], shrink=0.6)
# cbar.set_label(label='GCS    <---->    Sampling\nt-value (significant with alpha = 0.05)', size=18)
cbar.set_label(label='GCS    <---->    Sampling\nmedian across subjects', size=18)
# set max 3 yticks
cbar.ax.yaxis.set_major_locator(MaxNLocator(nbins=3))


# x_vals = gcs.value.values
# # y_vals = spo.value.values 
# y_vals = diff_df.value.values

# ax[-1].scatter(x_vals, y_vals, c=[channel_names.index(x) for x in gcs.channel.values], cmap='viridis')

x_vals = gcs.groupby('channel').value # .mean().values
y_vals = diff_df.groupby('channel').value # .mean().values

for i, channel in enumerate(gcs.channel.unique()):
    mask = np.array([j == i for j in range(len(gcs.channel.unique()))])
    ax[-1].scatter(x_vals.mean().values[mask], y_vals.mean().values[mask], c=[tab20c.colors[i]], label=channel, s=80)

lin_reg = LinearRegression().fit(x_vals.mean().values.reshape(-1, 1), y_vals.mean().values.reshape(-1, 1))
x = np.linspace(np.min(x_vals.mean().values), np.max(x_vals.mean().values), 100).reshape(-1, 1)
y = lin_reg.predict(x)

ax[-1].plot(x, y, color='red', linewidth=2, label=f'Linear fit:\n{lin_reg.coef_[0][0]:.3f}x + {lin_reg.intercept_[0]:.3f}')
ax[-1].legend(ncols=3)

ax[-1].set_xlabel('GCS')
ax[-1].set_ylabel('Difference in encoding performance (r)\n(Sampling - GCS)')
ax[-1].spines[['top', 'right']].set_visible(False)
# set maximally 4 yticks
ax[-1].yaxis.set_major_locator(MaxNLocator(nbins=4))
ax[-1].xaxis.set_major_locator(MaxNLocator(nbins=4))
ax[-1].axhline(0, color='gray', linestyle='--', linewidth=1)

ax[0].set_title('Best encoding model', fontsize=30, y=1.05)
ax[1].set_title('GCS vs. Sampling', fontsize=30, y=1.05)
ax[-1].set_title('Electrode-wise comparison', fontsize=30, y=1.05)
# fig.text(0.08, 0.99, 'Best encoding model', size=30, ha='left', transform=fig.transFigure)
# fig.text(0.35, 0.99, 'GCS vs. Sampling', size=30, ha='left', transform=fig.transFigure)
# fig.text(0.7, 0.99, 'Electrode-wise comparison', size=30, ha='left', transform=fig.transFigure)


fig.text(0.0, 1.02, 'A', size=48, weight='bold', ha='left', transform=fig.transFigure)
fig.text(0.3, 1.02, 'B', size=48, weight='bold', ha='left', transform=fig.transFigure)
fig.text(0.59, 1.02, 'C', size=48, weight='bold', ha='left', transform=fig.transFigure)

# fig.savefig(os.path.join(figure_dir, 'Fig-7_sampling_performance_fig7.pdf'), dpi=300, bbox_inches='tight')

plt.show()